In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

# Load model
llm = ChatOllama(model="qwen2.5:latest")

print("Chatbot Started! Type 'exit' to quit.\n")

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Bot: Goodbye!")
        break

    response = llm.invoke(
        [HumanMessage(content=user_input)]
    )

    print("Bot:", response.content)

In [ ]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt


class State(TypedDict):
    email: str
    approved: bool


def draft_email(state: State):
    print("\nDraft Email:")
    print(state["email"])

    # Pause graph and wait for human input
    approved = interrupt(
        "Do you approve this email? (yes/no)"
    )

    return {"approved": approved.lower() == "yes"}


def send_email(state: State):
    print("\n✅ Email sent!")
    return {}


def reject_email(state: State):
    print("\n❌ Email rejected!")
    return {}


def route(state: State):
    if state["approved"]:
        return "send"
    return "reject"


builder = StateGraph(State)

builder.add_node("draft", draft_email)
builder.add_node("send", send_email)
builder.add_node("reject", reject_email)

builder.add_edge(START, "draft")

builder.add_conditional_edges(
    "draft",
    route,
    {
        "send": "send",
        "reject": "reject",
    },
)

builder.add_edge("send", END)
builder.add_edge("reject", END)

graph = builder.compile()

config = {
    "configurable": {
        "thread_id": "email-approval"
    }
}

# Start execution
result = graph.invoke(
    {
        "email": "Dear Professor, may I have a 2-day extension?",
        "approved": False,
    },
    config=config,
)

print(result)